# Assignment 1 - Part 2: Software Effort Estimation

This notebook implements a reproducible workflow to:
- Use two public software effort datasets (NASA93 and COC81)
- Develop and compare multiple estimation models
- Apply rigorous validation with nested repeated cross-validation
- Evaluate with MAE, RMSE, MdAE, MASE, and MdASE
- Analyze performance and stability across datasets/folds

## 1. Setup

Run this notebook from the project folder `assignment1_part2_effort_estimation/`.
It will download raw data snapshots into `data/raw/` and create cleaned CSV files in `data/processed/`.

In [ ]:
from __future__ import annotations

import io
import json
import warnings
from pathlib import Path
from typing import Dict, Iterable, Tuple
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import GridSearchCV, KFold, RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Reproducibility constants
RANDOM_STATE = 42
OUTER_SPLITS = 5
OUTER_REPEATS = 10
INNER_SPLITS = 3
EPSILON = 1e-8

MIN_DATASETS = 2
MAX_DATASETS = 2

BASE_DIR = Path.cwd()
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIGURES_DIR = BASE_DIR / "figures"

DATASET_CANDIDATES = [
    {
        "name": "NASA93",
        "url": "https://zenodo.org/records/268419/files/nasa93.arff?download=1",
        "raw_name": "nasa93.arff",
        "format": "arff",
        "target": "act_effort",
        "drop": ["recordnumber"],
    },
    {
        "name": "China",
        "url": "https://zenodo.org/records/268446/files/china.arff?download=1",
        "raw_name": "china.arff",
        "format": "arff",
        "target": "Effort",
        "drop": ["ID", "N_effort"],
    },
    {
        "name": "COC81",
        "url": "https://zenodo.org/records/268424/files/coc81-dem.arff?download=1",
        "raw_name": "coc81-dem.arff",
        "format": "arff",
        "target": "effort",
        "drop": ["id", "defects"],
    },
]

for directory in [DATA_RAW_DIR, DATA_PROCESSED_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)



## 2. Data Acquisition and Loading

The helpers below snapshot candidate datasets and load them with a robust fallback strategy.
They try `NASA93`, then `China`, then `COC81`, skipping failures until at least two datasets are loaded.
Compatibility note: PROMISE ARFF variants are normalized for non-standard headers and whitespace-delimited rows.



In [ ]:
RATING_ORDER = ["vl", "l", "n", "h", "vh", "xh"]
RATING_SET = set(RATING_ORDER)


def download_dataset(url: str, out_path: Path) -> Path:
    """Download a file only if it is not present locally."""
    if not out_path.exists():
        print(f"Downloading {out_path.name}...")
        urlretrieve(url, out_path)
    else:
        print(f"Using cached file: {out_path.name}")
    return out_path


def _sanitize_arff_text(raw_text: str) -> str:
    """
    Normalize non-standard PROMISE headers such as:
    @class months numeric  ->  @attribute months numeric
    """
    cleaned = []
    for line in raw_text.splitlines():
        stripped = line.strip()
        if stripped.lower().startswith("@class "):
            leading_ws = line[: len(line) - len(line.lstrip())]
            cleaned.append(f"{leading_ws}@attribute {stripped[7:]}")
        else:
            cleaned.append(line)
    return "\n".join(cleaned) + "\n"


def _split_arff_sections(arff_text: str) -> tuple[list[str], list[tuple[int, str]]]:
    """Return header lines and (line_no, data_line) for non-empty data rows."""
    lines = arff_text.splitlines()
    header_lines = []
    data_lines: list[tuple[int, str]] = []
    in_data = False

    for i, line in enumerate(lines, start=1):
        stripped = line.strip()
        if not in_data:
            header_lines.append(line)
            if stripped.lower() == "@data":
                in_data = True
            continue

        if not stripped or stripped.startswith("%"):
            continue
        data_lines.append((i, stripped))

    return header_lines, data_lines


def _detect_data_delimiter(data_lines: list[tuple[int, str]]) -> str:
    if not data_lines:
        return "comma"
    return "comma" if "," in data_lines[0][1] else "whitespace"


def _parse_attribute_definition(line: str) -> dict:
    """Parse one ARFF @attribute definition."""
    raw = line.strip()[len("@attribute "):].strip()
    parts = raw.split(None, 1)
    if len(parts) < 2:
        raise ValueError(f"Malformed @attribute line: {line}")

    name = parts[0].strip("'\"")
    type_spec = parts[1].strip()
    lower_type = type_spec.lower()

    if type_spec.startswith("{") and type_spec.endswith("}"):
        kind = "nominal"
        values = [v.strip() for v in type_spec[1:-1].split(",")]
    elif lower_type in {"numeric", "real", "integer"}:
        kind = "numeric"
        values = None
    else:
        kind = "string"
        values = None

    return {"name": name, "type_spec": type_spec, "kind": kind, "values": values}


def _extract_attributes(header_lines: list[str]) -> list[dict]:
    attrs = []
    for line in header_lines:
        if line.strip().lower().startswith("@attribute "):
            attrs.append(_parse_attribute_definition(line))
    return attrs


def _tokenize_data_row(raw_row: str, delimiter: str) -> list[str]:
    if delimiter == "comma":
        return [token.strip() for token in raw_row.split(",")]
    return raw_row.split()


def load_arff(path: Path) -> pd.DataFrame:
    """Robust ARFF loader supporting comma and whitespace data rows."""
    raw_text = path.read_text(encoding="utf-8", errors="ignore")
    raw_lower = raw_text.lower()
    if "@relation" not in raw_lower or "@data" not in raw_lower:
        raise ValueError(f"File does not look like ARFF: {path}")

    sanitized_text = _sanitize_arff_text(raw_text)
    header_lines, data_lines = _split_arff_sections(sanitized_text)
    attributes = _extract_attributes(header_lines)
    if not attributes:
        raise ValueError(f"No @attribute definitions found in {path}")

    delimiter = _detect_data_delimiter(data_lines)
    expected_fields = len(attributes)

    rows = []
    for line_no, raw_row in data_lines:
        tokens = _tokenize_data_row(raw_row, delimiter)
        if len(tokens) != expected_fields:
            raise ValueError(
                f"ARFF row width mismatch at line {line_no}: "
                f"expected {expected_fields}, got {len(tokens)}. Row: {raw_row}"
            )
        rows.append(tokens)

    df = pd.DataFrame(rows, columns=[attr["name"] for attr in attributes])

    # Normalize missing markers and cast numeric columns.
    df = df.replace("?", np.nan)
    for attr in attributes:
        if attr["kind"] == "numeric":
            df[attr["name"]] = pd.to_numeric(df[attr["name"]], errors="coerce")

    return df


def snapshot_and_load(dataset_name: str, cfg: dict) -> pd.DataFrame:
    raw_path = DATA_RAW_DIR / cfg["raw_name"]
    download_dataset(cfg["url"], raw_path)

    dataset_format = cfg.get("format", "arff").lower()
    if dataset_format != "arff":
        raise ValueError(f"Unsupported dataset format: {dataset_format}")

    df = load_arff(raw_path)
    out_csv = DATA_PROCESSED_DIR / f"{dataset_name.lower()}_clean.csv"
    df.to_csv(out_csv, index=False)
    return df


def _validate_dataset_for_modeling(df: pd.DataFrame, cfg: dict) -> None:
    target = cfg["target"]
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found")
    if len(df) < 10:
        raise ValueError("Dataset has too few rows for cross-validation")


def load_available_datasets(
    candidates: list[dict],
    min_required: int = 2,
    max_selected: int = 2,
) -> tuple[dict[str, dict], pd.DataFrame]:
    """
    Attempt candidate datasets in order and keep successful loads.
    Returns selected datasets map and a load report dataframe.
    """
    selected: dict[str, dict] = {}
    report_rows = []

    for cfg in candidates:
        name = cfg["name"]
        try:
            df = snapshot_and_load(name, cfg)
            _validate_dataset_for_modeling(df, cfg)

            selected[name] = {"cfg": cfg, "df": df}
            report_rows.append(
                {
                    "dataset": name,
                    "status": "loaded",
                    "rows": int(df.shape[0]),
                    "columns": int(df.shape[1]),
                    "error": "",
                }
            )
        except Exception as exc:
            report_rows.append(
                {
                    "dataset": name,
                    "status": "failed",
                    "rows": 0,
                    "columns": 0,
                    "error": str(exc),
                }
            )

        if len(selected) >= max_selected:
            break

    report_df = pd.DataFrame(report_rows)
    report_df.to_csv(BASE_DIR / "dataset_load_report.csv", index=False)

    if len(selected) < min_required:
        loaded = list(selected.keys())
        raise RuntimeError(
            f"Only {len(selected)} dataset(s) loaded ({loaded}). "
            f"At least {min_required} required. See dataset_load_report.csv for details."
        )

    return selected, report_df



In [ ]:
selected_datasets, dataset_load_report = load_available_datasets(
    DATASET_CANDIDATES,
    min_required=MIN_DATASETS,
    max_selected=MAX_DATASETS,
)

selected_dataset_configs = {name: payload["cfg"] for name, payload in selected_datasets.items()}
datasets_df = {name: payload["df"] for name, payload in selected_datasets.items()}

print("Dataset load report:")
display(dataset_load_report)

print("Selected datasets:", list(datasets_df.keys()))
for name, df in datasets_df.items():
    print(f"{name}: shape={df.shape}")
    display(df.head())



## 3. Preprocessing Design

- Numerical variables: median imputation
- Ordered categorical ratings (e.g., `vl < l < n < h < vh < xh`): ordinal encoding
- Remaining categorical variables: one-hot encoding
- Target skewness control: models are trained on `log1p(y)` and predictions are converted back with `expm1`

In [ ]:
def split_feature_types(X: pd.DataFrame) -> Tuple[list, list, list]:
    """
    Return numeric, ordinal-categorical, and nominal-categorical columns.
    A categorical column is treated as ordinal when all observed values belong to
    the standard COCOMO rating scale.
    """
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    ordinal_cols = []
    nominal_cols = []

    for col in categorical_cols:
        values = set(str(v) for v in X[col].dropna().unique())
        if values and values.issubset(RATING_SET):
            ordinal_cols.append(col)
        else:
            nominal_cols.append(col)

    return numeric_cols, ordinal_cols, nominal_cols


def build_preprocessor(X: pd.DataFrame, for_linear: bool) -> ColumnTransformer:
    numeric_cols, ordinal_cols, nominal_cols = split_feature_types(X)

    transformers = []

    if numeric_cols:
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
        if for_linear:
            numeric_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(steps=numeric_steps), numeric_cols))

    if ordinal_cols:
        transformers.append(
            (
                "ord",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OrdinalEncoder(
                                categories=[RATING_ORDER] * len(ordinal_cols),
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                            ),
                        ),
                    ]
                ),
                ordinal_cols,
            )
        )

    if nominal_cols:
        transformers.append(
            (
                "nom",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                nominal_cols,
            )
        )

    return ColumnTransformer(transformers=transformers, remainder="drop")



## 4. Models and Hyperparameter Spaces

The workflow compares:
1. `ElasticNet` (regression-based requirement)
2. `RandomForestRegressor`
3. `GradientBoostingRegressor`

In [ ]:
def make_model_pipelines(X: pd.DataFrame, random_state: int = 42) -> Dict[str, Tuple[Pipeline, dict]]:
    pre_linear = build_preprocessor(X, for_linear=True)
    pre_tree = build_preprocessor(X, for_linear=False)

    elastic_pipeline = Pipeline(
        steps=[
            ("preprocessor", pre_linear),
            ("model", ElasticNet(max_iter=20000, random_state=random_state)),
        ]
    )

    rf_pipeline = Pipeline(
        steps=[
            ("preprocessor", pre_tree),
            ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1)),
        ]
    )

    gbr_pipeline = Pipeline(
        steps=[
            ("preprocessor", pre_tree),
            ("model", GradientBoostingRegressor(random_state=random_state)),
        ]
    )

    model_specs = {
        "ElasticNet": (
            elastic_pipeline,
            {
                "model__alpha": [0.001, 0.01, 0.1, 1.0],
                "model__l1_ratio": [0.1, 0.5, 0.9],
            },
        ),
        "RandomForest": (
            rf_pipeline,
            {
                "model__n_estimators": [300, 600],
                "model__max_depth": [None, 8, 16],
                "model__min_samples_leaf": [1, 3, 5],
                "model__max_features": ["sqrt", 0.6],
            },
        ),
        "GradientBoosting": (
            gbr_pipeline,
            {
                "model__n_estimators": [200, 400],
                "model__learning_rate": [0.03, 0.05, 0.1],
                "model__max_depth": [2, 3, 4],
                "model__subsample": [0.7, 1.0],
            },
        ),
    }

    return model_specs

## 5. Metric Functions

`MASE` and `MdASE` are computed against a naive constant predictor based on the **training-fold median effort**.

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_train_fold: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_train_fold = np.asarray(y_train_fold, dtype=float)

    abs_errors = np.abs(y_true - y_pred)

    train_median = np.median(y_train_fold)
    naive_train_errors = np.abs(y_train_fold - train_median)

    naive_mae_scale = max(np.mean(naive_train_errors), EPSILON)
    naive_mdae_scale = max(np.median(naive_train_errors), EPSILON)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mdae = median_absolute_error(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MdAE": mdae,
        "MASE": mae / naive_mae_scale,
        "MdASE": mdae / naive_mdae_scale,
    }

## 6. Nested Repeated Cross-Validation Evaluation

- Outer loop: `RepeatedKFold(5 splits, 10 repeats)` for robust generalization estimation.
- Inner loop: `KFold(3)` inside each outer training fold for hyperparameter tuning.
- Scoring during tuning: negative MAE on log-transformed target.

In [ ]:
def evaluate_nested_cv(
    X: pd.DataFrame,
    y: pd.Series,
    dataset_name: str,
    random_state: int = 42,
) -> pd.DataFrame:
    model_specs = make_model_pipelines(X, random_state=random_state)

    outer_cv = RepeatedKFold(
        n_splits=OUTER_SPLITS,
        n_repeats=OUTER_REPEATS,
        random_state=random_state,
    )

    rows = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx].astype(float)
        y_test = y.iloc[test_idx].astype(float)

        y_train_log = np.log1p(y_train)

        inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=random_state)

        for model_name, (pipeline, param_grid) in model_specs.items():
            gs = GridSearchCV(
                estimator=clone(pipeline),
                param_grid=param_grid,
                cv=inner_cv,
                scoring="neg_mean_absolute_error",
                n_jobs=-1,
                refit=True,
            )
            gs.fit(X_train, y_train_log)

            y_pred_log = gs.best_estimator_.predict(X_test)
            y_pred = np.expm1(y_pred_log)
            y_pred = np.clip(y_pred, a_min=0, a_max=None)

            metrics = compute_metrics(y_test.to_numpy(), y_pred, y_train.to_numpy())

            row = {
                "dataset": dataset_name,
                "fold": fold_id,
                "model": model_name,
                "best_params": json.dumps(gs.best_params_),
            }
            row.update(metrics)
            rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
all_results = []

for dataset_name, cfg in selected_dataset_configs.items():
    df = datasets_df[dataset_name].copy()

    target_col = cfg["target"]
    drop_cols = [c for c in cfg["drop"] if c in df.columns]

    X = df.drop(columns=[target_col] + drop_cols)
    y = df[target_col].astype(float)

    print(f"Evaluating {dataset_name}: X={X.shape}, y={y.shape}")
    result_df = evaluate_nested_cv(X, y, dataset_name=dataset_name, random_state=RANDOM_STATE)
    all_results.append(result_df)

results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv(BASE_DIR / "results_by_fold.csv", index=False)

print("Saved fold-level metrics to results_by_fold.csv")
results_df.head()



## 7. Aggregation and Stability Analysis

In [ ]:
def iqr(series: pd.Series) -> float:
    return float(series.quantile(0.75) - series.quantile(0.25))

metric_cols = ["MAE", "RMSE", "MdAE", "MASE", "MdASE"]

summary_df = (
    results_df
    .groupby(["dataset", "model"], as_index=False)[metric_cols]
    .agg(["mean", "median", "std", iqr])
)
summary_df.columns = ["_".join(col).strip("_") for col in summary_df.columns]
summary_df = summary_df.rename(columns={"dataset_": "dataset", "model_": "model"})

summary_df.to_csv(BASE_DIR / "results_summary.csv", index=False)
print("Saved aggregate metrics to results_summary.csv")

summary_df

In [ ]:
rank_table = (
    results_df
    .groupby(["dataset", "model"], as_index=False)[metric_cols]
    .mean()
)

for metric in metric_cols:
    rank_table[f"rank_{metric}"] = rank_table.groupby("dataset")[metric].rank(method="dense", ascending=True)

rank_table = rank_table.sort_values(["dataset", "rank_MAE", "rank_RMSE"])
rank_table.to_csv(BASE_DIR / "model_ranking.csv", index=False)
rank_table

## 8. Visualization

In [ ]:
for metric in metric_cols:
    plt.figure()
    ax = sns.boxplot(data=results_df, x="model", y=metric, hue="dataset")
    ax.set_title(f"{metric} distribution across outer folds")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"boxplot_{metric.lower()}.png", dpi=150)
    plt.show()

In [ ]:
stability_rows = []
for (dataset_name, model_name), g in results_df.groupby(["dataset", "model"]):
    for metric in metric_cols:
        m = g[metric].mean()
        s = g[metric].std(ddof=0)
        cv = s / (m + EPSILON)
        stability_rows.append(
            {
                "dataset": dataset_name,
                "model": model_name,
                "metric": metric,
                "mean": m,
                "std": s,
                "coef_var": cv,
            }
        )

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(BASE_DIR / "stability_metrics.csv", index=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=stability_df, x="metric", y="coef_var", hue="model")
ax.set_title("Model stability (coefficient of variation) by metric")
ax.set_ylabel("Coefficient of variation (lower is more stable)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stability_coefvar.png", dpi=150)
plt.show()

## 9. Report Support Outputs

The cells below export compact markdown tables that can be copied directly into the technical report draft.

In [ ]:
summary_for_report = (
    results_df
    .groupby(["dataset", "model"], as_index=False)[metric_cols]
    .mean()
    .round(4)
    .sort_values(["dataset", "MAE", "RMSE"])
)

markdown_table = summary_for_report.to_markdown(index=False)
print(markdown_table)

with open(BASE_DIR / "summary_table.md", "w", encoding="utf-8") as f:
    f.write(markdown_table + "\n")

print("Saved summary markdown table to summary_table.md")



## 10. Interpretation Checklist

Use this section to complete the final narrative after execution:
- Which model has the lowest average MAE/RMSE per dataset?
- Is ranking consistent across MAE, MdAE, MASE, MdASE?
- Which model has lower fold variance (stability)?
- Do NASA93 and COC81 favor the same model family?
- How do skewness, sample size, and categorical structure explain observed behavior?

## 11. References

- NASA93 dataset (PROMISE/Zenodo): https://zenodo.org/records/268419
- China dataset (PROMISE/Zenodo): https://zenodo.org/records/268446
- COC81 DEM dataset (PROMISE/Zenodo): https://zenodo.org/records/268424
- Assignment metrics guidance: MAE, RMSE, MdAE, MASE, MdASE

